In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install py_vncorenlp underthesea pyserini faiss-cpu

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 5.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.2/412.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.7/196.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os

def install_java():
  !apt update -qq
  !apt-get install -y openjdk-21-jdk-headless -qq > /dev/null
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
  !java -version

install_java()

6 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
!git clone https://github.com/tommachilez/virag4fc.git
%cd /content/virag4fc/

Cloning into 'fact-checking-data-framework-vn'...
remote: Enumerating objects: 561, done.
remote: Counting objects: 100% (217/217), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 561 (delta 144), reused 114 (delta 62), pack-reused 344 (from 1)
Receiving objects: 100% (561/561), 5.36 MiB | 6.41 MiB/s, done.
Resolving deltas: 100% (368/368), done.
/content/fact-checking-data-framework-vn


In [ ]:
VNCORE_MODEL_PATH = "/content/drive/MyDrive/KLTN_Project/Models/vncorenlp_models"
dataset_path = "/content/drive/MyDrive/KLTN_Project/Datasets"
csv_path = f"{dataset_path}/vifc_test_set.csv"
stopwords_path = f"{dataset_path}/stopwords-vi.txt"

# BM25 Hard Negative Mining

In [ ]:
mining_dir = f"{dataset_path}/vn_mining_test"
training_triples = f"{dataset_path}/test_triples.jsonl"
doc_mapping = f"{dataset_path}/unique_doc_mapping.csv"

In [ ]:
# Step 1: Preprocessing (Uses PyVnCoreNLP)
!python -m src.scripts.hard_negative_mining.preprocess_csv \
    --input_csv {csv_path} \
    --doc_mapping {doc_mapping} \
    --vncorenlp_path {VNCORE_MODEL_PATH} \
    --stopwords_path {stopwords_path} \
    --output_dir {mining_dir}

2025-12-25 13:06:37 INFO  WordSegmenter:24 - Loading Word Segmentation model
>>> Processing Document Mapping: /content/drive/MyDrive/KLTN_Project/Datasets/unique_doc_mapping.csv
Tokenizing Corpus: 3030it [00:46, 65.77it/s]
>>> Processing Input Queries: /content/drive/MyDrive/KLTN_Project/Datasets/vifc_test_set.csv
Processing Queries: 7687it [00:10, 714.04it/s]
>>> Preprocessing complete.
    Files saved to /content/drive/MyDrive/KLTN_Project/Datasets/vn_mining_test
    Queries Processed: 7687
    Skipped (Doc text not found in mapping): 0


In [ ]:
# Step 2: Mining (Uses Pyserini)
# Note: This is a separate process execution, so a new JVM is started.
!python -m src.scripts.hard_negative_mining.pyserini_mining \
    --preprocessed_dir {mining_dir} \
    --doc_mapping {doc_mapping} \
    --output_jsonl {training_triples} \
    --top_k 10

2025-12-25 13:19:18.609637: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-25 13:19:18.615324: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-25 13:19:18.629532: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766668758.654048    5119 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766668758.661016    5119 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766668758.679418    5119 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
import json
import os

class DatasetViewer:
    def __init__(self, corpus_path: str, queries_path: str, triples_path: str):
        """
        Initialize with paths to the three key files.
        """
        self.paths = {
            "corpus": corpus_path,
            "queries": queries_path,
            "triples": triples_path
        }

    def _view_file(self, key: str, n: int):
        path = self.paths.get(key)
        title = key.replace('_', ' ').title()

        print(f"\n{'='*20} VIEWING: {title} (First {n} lines) {'='*20}")

        if not path or not os.path.exists(path):
            print(f"❌ File not found at: {path}")
            return

        try:
            with open(path, 'r', encoding='utf-8') as f:
                for i in range(n):
                    line = f.readline()
                    if not line:
                        print(f"--- End of file reached at line {i} ---")
                        break

                    try:
                        # Parse JSON to pretty-print it
                        data = json.loads(line.strip())
                        print(f"🔹 [Line {i+1}]")
                        print(json.dumps(data, indent=2, ensure_ascii=False))
                    except json.JSONDecodeError:
                        print(f"⚠️ [Line {i+1}] (Invalid JSON)")
                        print(line.strip())
        except Exception as e:
            print(f"Error reading file: {e}")

    def view_corpus(self, n: int = 5):
        """View the pretokenized corpus file."""
        self._view_file("corpus", n)

    def view_queries(self, n: int = 5):
        """View the pretokenized queries file."""
        self._view_file("queries", n)

    def view_triples(self, n: int = 5):
        """View the final output training triples."""
        self._view_file("triples", n)

    def view_all(self, n: int = 3):
        """View first n lines of all files sequentially."""
        self.view_corpus(n)
        self.view_queries(n)
        self.view_triples(n)

In [ ]:
viewer = DatasetViewer(
    f"{mining_dir}/corpus_pretokenized.jsonl",
    f"{mining_dir}/queries_pretokenized.jsonl",
    training_triples,
)

In [ ]:
viewer.view_all()


==================== VIEWING: Corpus (First 3 lines) ====================
🔹 [Line 1]
{
  "id": "0",
  "contents": "chinhphu.vn mong_muốn gửi_gắm phó thủ_tướng trần hồng hà truyền_hình lễ bế_mạc liên_hoan truyền_hình toàn_quốc thứ 41 tối 18/3 tp hải_phòng phó thủ_tướng trần hồng hà tác_phẩm truyền_hình vun_đắp làm_giàu văn_hoá việt_nam tiên_tiến đậm_đà bản_sắc dân_tộc góp_phần tạo_dựng môi_trường văn_hoá lành_mạnh xây_dựng con_người việt_nam nhân_cách trách_nhiệm hội_nhập ảnh vgp minh khôi tham_dự lễ bế_mạc bí_thư trung_ương đảng trưởng ban tuyên_giáo trung_ương nguyễn_trọng nghĩa lãnh_đạo ngành trung_ương địa_phương đại_diện đài_truyền_hình đơn_vị sản_xuất chương_trình truyền_hình đông_đảo cán_bộ phóng_viên biên_tập_viên nghệ_sĩ diễn_viên hoạt_động lĩnh_vực truyền_hình … thay_mặt chính_phủ thủ_tướng chính_phủ phó thủ_tướng trần hồng hà chúc_mừng đài_truyền_hình việt_nam đài_truyền_hình tỉnh thành_phố nước đơn_vị sản_xuất truyền_hình tp hải_phòng tổ_chức thành_công sự_kiện quan_trọng 2